# Evaluate Medical Models on General Datasets
Models to evaluate (in order):
1. `google/medgemma-4b-it`
2. `microsoft/llava-med-v1.5-mistral-7b`
3. `FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL`

Datasets: `lmms-lab/VQAv2` (sampled), `lmms-lab/OK-VQA` (sampled)

Note: For 7B models on Kaggle T4, we use 4-bit quantization via bitsandbytes.\n

In [1]:
# Kaggle setup: Nuke broken versions, force stable Pillow
!pip uninstall -y transformers pillow
!pip install -q --upgrade transformers datasets bitsandbytes accelerate qwen-vl-utils "pillow<12.0"

import os, json, re, string, random, gc
import torch
from PIL import Image
from datasets import load_dataset
from tqdm.auto import tqdm

# This should now import perfectly!
from transformers import BitsAndBytesConfig, AutoProcessor, AutoModelForImageTextToText

print('torch:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 58.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 require

In [2]:
# ── Image utilities ───────────────────────────────────────────────
def to_rgb(img: Image.Image) -> Image.Image:
    return img if img.mode == 'RGB' else img.convert('RGB')

# ── Final Answer extraction (same as 03_inference_harness_v2 style) ─
def extract_final_answer(text: str) -> str:
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', text, re.DOTALL)
    if match:
        ans = match.group(1).strip()
        ans = re.sub(r'[\*"\']+', '', ans).strip()
        ans = ans.split('\n')[0].strip()
        return ans
    # fallback: first sentence
    return re.split(r'(?<=[.!?])\s', text)[0].strip()

# ── Prompt builders (MedGemma protocol for VQAv2/OK-VQA v2) ────────
def build_prompt_vqav2(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}{question} "
        "You may write out your argument before stating your final very short, "
        "definitive, and concise answer (if possible, a single word) "
        "X in the format 'Final Answer: X'"
    )

def build_prompt_okvqa(question: str, is_closed: bool) -> str:
    return (
        f"{question} "
        "You may write out your argument before stating your final very short, "
        "definitive, and concise answer (if possible, a single word) "
        "X in the format 'Final Answer: X'"
    )

def get_vqav2_answer(sample):
    answers = [a['answer'].strip().lower() for a in sample['answers']]
    return max(set(answers), key=answers.count)

def is_vqav2_closed(sample):
    ans = get_vqav2_answer(sample)
    return ans in ('yes', 'no')


In [3]:
# ── Dataset loading (streaming) ───────────────────────────────────
print('Streaming OK-VQA...')
okvqa_stream = load_dataset('lmms-lab/OK-VQA', split='val2014', streaming=True)
okvqa_test = []
for s in okvqa_stream:
    if len(okvqa_test) >= 1000:
        break
    okvqa_test.append(s)
print('OK-VQA sampled:', len(okvqa_test))

Streaming OK-VQA...


README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

OK-VQA sampled: 1000


In [4]:
# ── Runner ─────────────────────────────────────────────────────────
def run_dataset(model, processor, samples, dataset_name, model_name, is_qwen=False,
                 get_image_fn=None, get_question_fn=None, get_answer_fn=None, get_is_closed_fn=None,
                 build_prompt_fn=None,
                 output_dir='./outputs', max_new_tokens=100):

    os.makedirs(output_dir, exist_ok=True)
    safe_model = model_name.replace('/', '_')
    out_path = os.path.join(output_dir, f'{safe_model}__{dataset_name}_v2.jsonl')

    completed = set()
    if os.path.exists(out_path):
        with open(out_path, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                completed.add(r['idx'])
        print('Resuming:', len(completed), 'samples already done')

    errors = 0
    with open(out_path, 'a') as f_out:
        for i, sample in enumerate(tqdm(samples, desc=f'{model_name} | {dataset_name}')):
            if i in completed:
                continue

            try:
                image = to_rgb(get_image_fn(sample))
                question = get_question_fn(sample)
                answer = get_answer_fn(sample)
                is_closed = get_is_closed_fn(sample)
                prompt_text = build_prompt_fn(question, is_closed)

                messages = [{
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': image},
                        {'type': 'text', 'text': prompt_text},
                    ]
                }]

                text = processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )

                # Processor APIs differ slightly between models; handle common cases
                if is_qwen:
                    inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to(device)
                    input_len = inputs['input_ids'].shape[-1]
                else:
                    inputs = processor(text=text, images=image, return_tensors='pt').to(device)
                    input_len = inputs['input_ids'].shape[-1]

                with torch.inference_mode():
                    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

                if is_qwen:
                    # Qwen-style: decode after input_len
                    raw = processor.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)[0].strip()
                else:
                    raw = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()

                prediction = extract_final_answer(raw)

                record = {
                    'idx': i,
                    'question': question,
                    'ground_truth': answer,
                    'prediction': prediction,
                    'raw_output': raw,
                    'is_closed': is_closed,
                    'model': model_name,
                    'dataset': dataset_name,
                }
            except Exception as e:
                errors += 1
                record = {
                    'idx': i,
                    'question': '',
                    'ground_truth': '',
                    'prediction': '',
                    'raw_output': '',
                    'is_closed': False,
                    'model': model_name,
                    'dataset': dataset_name,
                    'error': str(e)
                }

            f_out.write(json.dumps(record) + '\n')
            f_out.flush()

    print(f'Done. Output saved to {out_path}. Errors: {errors}')
    return out_path


In [5]:
# Hugging-face-login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret('HF_TOKEN'))
print('Logged in.')


Logged in.


In [6]:
# ── Model 1: MedGemma (4B) ─────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import gc

medgemma_model_id = 'google/medgemma-4b-it'
print(f'Loading {medgemma_model_id}...')

medgemma_processor = AutoProcessor.from_pretrained(medgemma_model_id)

medgemma_model = AutoModelForImageTextToText.from_pretrained(
    medgemma_model_id,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

# Run on OK-VQA
out_okvqa_medgemma = run_dataset(
    model=medgemma_model,
    processor=medgemma_processor,
    samples=okvqa_test,
    dataset_name='okvqa',
    model_name=medgemma_model_id,
    is_qwen=False,
    get_image_fn=lambda x: x['image'],
    get_question_fn=lambda x: x['question'],
    get_answer_fn=lambda x: max(
        set([a.strip().lower() for a in x['answers']]),
        key=[a.strip().lower() for a in x['answers']].count
    ),
    get_is_closed_fn=lambda x: False,
    build_prompt_fn=build_prompt_okvqa,
    output_dir='./outputs'
)

# Free memory before next model
del medgemma_model
del medgemma_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading google/medgemma-4b-it...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

google/medgemma-4b-it | okvqa:   0%|          | 0/1000 [00:00<?, ?it/s]

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


Done. Output saved to ./outputs/google_medgemma-4b-it__okvqa_v2.jsonl. Errors: 0


In [7]:
# ── Model 2: LLaVA-Med (7B) ───────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch
import gc

llava_model_id = 'chaoyinshe/llava-med-v1.5-mistral-7b-hf'
print(f'Loading {llava_model_id} in 4-bit...')

# Quantized 4-bit for Kaggle T4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

llava_processor = AutoProcessor.from_pretrained(llava_model_id)

llava_model = AutoModelForImageTextToText.from_pretrained(
    llava_model_id,
    device_map='auto',
    quantization_config=quantization_config
)

# Run on OK-VQA
out_okvqa_llava = run_dataset(
    model=llava_model,
    processor=llava_processor,
    samples=okvqa_test,
    dataset_name='okvqa',
    model_name=llava_model_id,
    is_qwen=False,
    get_image_fn=lambda x: x['image'],
    get_question_fn=lambda x: x['question'],
    get_answer_fn=lambda x: max(
        set([a.strip().lower() for a in x['answers']]),
        key=[a.strip().lower() for a in x['answers']].count
    ),
    get_is_closed_fn=lambda x: False,
    build_prompt_fn=build_prompt_okvqa,
    output_dir='./outputs'
)

# Free memory before next model
del llava_model
del llava_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading chaoyinshe/llava-med-v1.5-mistral-7b-hf in 4-bit...


processor_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

chaoyinshe/llava-med-v1.5-mistral-7b-hf | okvqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Done. Output saved to ./outputs/chaoyinshe_llava-med-v1.5-mistral-7b-hf__okvqa_v2.jsonl. Errors: 0


In [8]:
# ── Model 3: HuatuoGPT (Qwen2.5-VL) ───────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch
import gc

huatuo_model_id = 'FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL'
print(f'Loading {huatuo_model_id} in 4-bit...')

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

huatuo_processor = AutoProcessor.from_pretrained(huatuo_model_id)

huatuo_model = AutoModelForImageTextToText.from_pretrained(
    huatuo_model_id,
    device_map='auto',
    quantization_config=quantization_config
)

# Run on OK-VQA
out_okvqa_huatuo = run_dataset(
    model=huatuo_model,
    processor=huatuo_processor,
    samples=okvqa_test,
    dataset_name='okvqa',
    model_name=huatuo_model_id,
    is_qwen=True,
    get_image_fn=lambda x: x['image'],
    get_question_fn=lambda x: x['question'],
    get_answer_fn=lambda x: max(
        set([a.strip().lower() for a in x['answers']]),
        key=[a.strip().lower() for a in x['answers']].count
    ),
    get_is_closed_fn=lambda x: False,
    build_prompt_fn=build_prompt_okvqa,
    output_dir='./outputs'
)

print('All models evaluated successfully on OK-VQA.')
del huatuo_model
del huatuo_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL in 4-bit...


preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL | okvqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Done. Output saved to ./outputs/FreedomIntelligence_HuatuoGPT-Vision-7B-Qwen2.5VL__okvqa_v2.jsonl. Errors: 0
All models evaluated successfully on OK-VQA.
